# Approach 03 (fixed) - BI prune + real KD repair

In [ ]:
!pip install -q transformers datasets accelerate pandas matplotlib

In [ ]:
import torch, transformers, math, time, gc
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
import random, numpy as np
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_IDS=["openai-community/gpt2","HuggingFaceTB/SmolLM-135M"]
tokenizers={}
for mid in MODEL_IDS:
    tok=AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    tokenizers[mid]=tok

In [ ]:
from datasets import load_dataset
dataset=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1")
def join_nonempty(split): return "\n\n".join(x for x in split["text"] if x and x.strip())
train_text=join_nonempty(dataset["train"])[:300000]
validation_text=join_nonempty(dataset["validation"])[:100000]
test_text=join_nonempty(dataset["test"])
print(dataset)

In [ ]:
def get_layers(m):
    if hasattr(m,"transformer") and hasattr(m.transformer,"h"): return m.transformer.h
    return m.model.layers
def set_layers(m, blocks):
    blocks=torch.nn.ModuleList(blocks)
    if hasattr(m,"transformer") and hasattr(m.transformer,"h"):
        m.transformer.h=blocks; m.config.n_layer=len(blocks)
    else:
        m.model.layers=blocks; m.config.num_hidden_layers=len(blocks)
    return m
def load_fresh(mid):
    m=AutoModelForCausalLM.from_pretrained(mid); m.config.use_cache=False; m.eval(); return m
def free(m):
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
EVAL_TOKENS=3000
def calculate_ppl(model, text, mid, block=256, max_tokens=EVAL_TOKENS):
    model.eval(); tok=tokenizers[mid]
    ids=tok(text,return_tensors="pt",add_special_tokens=False)["input_ids"][0]
    if max_tokens is not None: ids=ids[:max_tokens]
    n=max(1,len(ids)//block); nll=0.0; ntok=0
    with torch.no_grad():
        for i in range(n):
            x=ids[i*block:(i+1)*block].unsqueeze(0).to(device)
            o=model(input_ids=x,labels=x,use_cache=False)
            nll+=o.loss.item()*(x.numel()-1); ntok+=x.numel()-1
    return math.exp(nll/ntok)

In [ ]:
import torch.nn.functional as F
def block_influence(m, mid, text, n=2000):
    tok=tokenizers[mid]; ids=tok(text[:n],return_tensors="pt",add_special_tokens=False).to(device)
    with torch.no_grad(): out=m(**ids,output_hidden_states=True,use_cache=False)
    hs=out.hidden_states
    return [1.0-F.cosine_similarity(hs[i].float(),hs[i+1].float(),dim=-1).mean().item() for i in range(len(hs)-1)]

In [ ]:
def hard_remove(mid, idx):
    m=load_fresh(mid); blocks=list(get_layers(m))
    kept=[b for i,b in enumerate(blocks) if i not in set(idx)]
    set_layers(m,kept)
    for i,l in enumerate(get_layers(m)):
        if hasattr(l,"layer_idx"): l.layer_idx=i
        if hasattr(l,"self_attn") and hasattr(l.self_attn,"layer_idx"): l.self_attn.layer_idx=i
    return m

In [ ]:
def kd_repair(student, teacher, mid, steps=200, lr=2e-5, seq_len=256, T=2.0):
    tok=tokenizers[mid]
    ids=tok(train_text[:40000],return_tensors="pt",add_special_tokens=False)["input_ids"][0]
    opt=torch.optim.AdamW(student.parameters(),lr=lr)
    student.train(); teacher.eval()
    for i in range(steps):
        start=(i*seq_len)%max(1,len(ids)-seq_len)
        x=ids[start:start+seq_len].unsqueeze(0).to(device)
        s_out=student(input_ids=x,labels=x,use_cache=False)
        with torch.no_grad(): t_out=teacher(input_ids=x,use_cache=False)
        s_log=F.log_softmax(s_out.logits/T,dim=-1)
        t_p=F.softmax(t_out.logits/T,dim=-1)
        kl=F.kl_div(s_log,t_p,log_target=False,reduction="batchmean")*(T*T)
        loss=s_out.loss + kl
        opt.zero_grad(); loss.backward(); opt.step()
        if i%50==0: print(i, round(loss.item(),4), "ce", round(s_out.loss.item(),4), "kl", round(kl.item(),4))
    return student

In [ ]:
rows=[]
for mid in MODEL_IDS:
    teacher=load_fresh(mid).to(device)
    m0=load_fresh(mid).to(device); bis=block_influence(m0,mid,train_text); free(m0)
    order=sorted(range(len(bis)), key=lambda i: bis[i])
    for k in [1,2]:
        target=sorted(order[:k])
        m=hard_remove(mid,target).to(device)
        pre=calculate_ppl(m,validation_text,mid)
        kd_repair(m,teacher,mid,steps=200)
        post=calculate_ppl(m,validation_text,mid)
        rows.append({"model":mid,"k":k,"ppl_before":pre,"ppl_after":post})
        print(mid,k,target,round(pre,2),"->",round(post,2))
        free(m)
    free(teacher)
import pandas as pd
pd.DataFrame(rows)